# Qwen3-4B Financial SFT — one-click repro

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hydaspex/qwen3-financial-sft/blob/main/notebooks/colab_repro.ipynb)

End-to-end repro of [qwen3-financial-sft](https://github.com/Hydaspex/qwen3-financial-sft): LoRA/QLoRA SFT of Qwen3-4B-Instruct on TAT-QA financial reasoning, with an offline eval harness (numeric-tolerance EM + span match).

**Runtime**: GPU required — `Runtime > Change runtime type > T4 GPU`. Quick-run defaults: 2,000 samples, 1 epoch (~30 min on T4).


In [ ]:
import torch

assert torch.cuda.is_available(), "Enable a GPU runtime: Runtime > Change runtime type > T4"
print(torch.cuda.get_device_name(0), "| bf16 supported:", torch.cuda.is_bf16_supported())


In [ ]:
!git clone -q https://github.com/Hydaspex/qwen3-financial-sft.git
%cd qwen3-financial-sft
!pip install -q -e .


## 1. Config — quick-run overrides

Free-tier Colab: subsample TAT-QA and train 1 epoch. The repo config stays unchanged for full local/Databricks runs.


In [ ]:
from finsft.config import load_config

cfg = load_config("configs/sft_lora_qwen3_4b.yaml")
cfg.data.max_samples = 2000  # quick run; None = full TAT-QA
cfg.trainer.num_train_epochs = 1
print(cfg.experiment_name, "| model:", cfg.model.name_or_path)


## 2. Data prep — TAT-QA → chat-format JSONL


In [ ]:
from datasets import load_dataset
from finsft.data import flatten_tatqa, train_val_split, write_jsonl

raw = load_dataset(cfg.data.dataset_name, split="train")
records = flatten_tatqa(raw, cfg.data.max_context_chars, cfg.data.max_samples)
train, val = train_val_split(records, cfg.data.val_fraction, cfg.seed)
write_jsonl(train, cfg.data.train_path)
write_jsonl(val, cfg.data.val_path)
print(f"{len(train)} train / {len(val)} val")
print(train[0]["messages"][1]["content"][:400])


## 3. Train — QLoRA SFT (TRL `SFTTrainer`)

4-bit NF4 quantisation, LoRA r=16 on all attention + MLP projections, completion-only loss. On T4 the trainer automatically falls back to fp16 + sdpa.


In [ ]:
from finsft.train import build_trainer

trainer = build_trainer(cfg)
trainer.train()
trainer.save_model(str(cfg.trainer.output_dir))


## 4. Evaluate — base vs fine-tuned

Numeric-tolerance exact match + span match on 50 held-out examples.


In [ ]:
import gc

from finsft.evaluate import generate_answers, score

sample = val[:50]
golds = [r["messages"][-1]["content"] for r in sample]

preds_base = generate_answers(cfg, sample, adapter_path=None)
gc.collect(); torch.cuda.empty_cache()  # free base model before adapter load
preds_lora = generate_answers(cfg, sample, adapter_path=str(cfg.trainer.output_dir))

print("base :", score(preds_base, golds))
print("tuned:", score(preds_lora, golds))


## Results

| Model | Numeric EM (tol=1e-3) | Span match |
|---|---|---|
| Qwen3-4B-Instruct (base) | _fill after run_ | _fill after run_ |
| + LoRA SFT | _fill after run_ | _fill after run_ |

Full runs (all of TAT-QA, 3 epochs) belong on a larger GPU or a Databricks job — see the repo README.
